In [ ]:
import numpy as np
import  pandas as pd
import warnings

warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('qoute_dataset.csv')

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
quotes[0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [ ]:
quotes = quotes.str.lower()

In [ ]:
quotes[0]

'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [ ]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [ ]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 10000

tokenizer = Tokenizer(num_words= vocab_size)
tokenizer.fit_on_texts(quotes)

In [ ]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [ ]:
sentences = tokenizer.texts_to_sequences(quotes)

In [ ]:
X = []
y = []

for seq in sentences:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)


In [ ]:
len(X)

85271

In [ ]:
len(y)

85271

In [ ]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_pad = pad_sequences(X, maxlen = max_len, padding= 'pre')

In [ ]:
X_pad

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [ ]:
y = np.array(y)

In [ ]:
X_pad.shape

(85271, 745)

In [ ]:
from tensorflow.keras.utils import to_categorical
y_onehot = to_categorical(y, num_classes= vocab_size )

In [ ]:
y_onehot.shape

(85271, 10000)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,SimpleRNN

In [ ]:
embedded_dim = 50
rnn_units = 128


In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedded_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedded_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 100
batch_size = 128


In [ ]:
# hist_rnn = rnn_model.fit(
#     X_pad, y_onehot, epochs=epochs, batch_size=batch_size, verbose=1,validation_split=0.2
# )

In [ ]:
hist_lstm = lstm_model.fit(
    X_pad, y_onehot, epochs=epochs, batch_size=batch_size, verbose=1,validation_split=0.2
)

Epoch 1/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.0391 - loss: 6.7623 - val_accuracy: 0.0440 - val_loss: 6.7595
Epoch 2/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 58ms/step - accuracy: 0.0545 - loss: 6.3393 - val_accuracy: 0.0579 - val_loss: 6.6770
Epoch 3/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 32s 60ms/step - accuracy: 0.0714 - loss: 6.1099 - val_accuracy: 0.0772 - val_loss: 6.5773
Epoch 4/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 33s 62ms/step - accuracy: 0.0912 - loss: 5.8916 - val_accuracy: 0.0901 - val_loss: 6.5253
Epoch 5/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 32s 59ms/step - accuracy: 0.1041 - loss: 5.7134 - val_accuracy: 0.0982 - val_loss: 6.5197
Epoch 6/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 32s 59ms/step - accuracy: 0.1134 - loss: 5.5549 - val_accuracy: 0.0993 - val_loss: 6.5253
Epoch 7/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 32s 59ms/step - accuracy: 0.1218 - loss: 5.4017 - val_accuracy: 0.1044 - val_loss: 6.5453
Epoch 8/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 59ms/step - accuracy: 0.1296 - loss: 5

In [ ]:
lstm_model.save('lstm_model.h5')

In [ ]:
index_to_word = {}
for word,index in word_index.items():
    index_to_word[index] = word

In [54]:
def predictor(model,text,tokenizer,max_len):

  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [58]:
seed_text = "what are you"
next_word = predictor(lstm_model,seed_text,tokenizer,max_len)
print(next_word)

lurking


In [59]:
def generate_text(model,seed_text,num_words,tokenizer,max_len):
  text = seed_text
  for i in range(num_words):
    next_word = predictor(model,text,tokenizer,max_len)
    if next_word == "":
      break
    text += ' ' + next_word
  return text

In [60]:
seed_text = 'life is a'
generate_text = generate_text(lstm_model,seed_text,10,tokenizer,max_len)
print(generate_text)

life is a series of natural and spontaneous changes dont resist them that
